# Misja 15: od planu do wykonania — most do drzew zachowań

Planer z ćwiczeń 12–14 wybiera **kolejność legalnych działań**. Drzewo zachowań z ćwiczeń 9–11
pilnuje wykonywania: czeka na wynik, reaguje na błąd i przerywa pracę. Połączymy te dwa poziomy na przykładzie restauracji.

**Cel:** odróżnić zaplanowany efekt od potwierdzonego wykonania, lokalną ponowną próbę od przeplanowania
oraz błąd działania od obserwacji zmienionego świata. Sukces jednego działania nie oznacza ukończenia całej misji.

To **model wykonania w czystym Pythonie**. Nie uruchamia ROS, Nav2 ani Gazebo i nie porusza robotem.
Wyniki są kontrolowane przez `Outcome`; tick jest krokiem testu, nie sekundą.
Możesz uruchomić wszystkie komórki przed rozwiązaniem zadania — zobaczysz oznaczenie TODO.

In [ ]:
from pathlib import Path
import sys

base = Path.cwd().resolve()
for candidate in (base, *base.parents, base / 'jupyter_notebooks'):
    if (candidate / 'robo_restaurant' / 'execution.py').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError('Otwórz notebook wewnątrz repozytorium')

from dataclasses import replace
from robo_restaurant.planning import State, Action, solve, execute, goal
from robo_restaurant.execution import ActionExecutor, Outcome, Status
print('Model gotowy: żadnych procesów ani wątków w tle.')

## 1. Kto odpowiada za co?

| Poziom | Decyzja | Co pozostaje do sprawdzenia? |
| --- | --- | --- |
| Planer | `solve(state)` wybiera działania i koszt | Czy robot cokolwiek wykonał |
| Executor | Wysyła jedno działanie, czeka na potwierdzenie | Czy reszta planu nadal pasuje do świata |
| Drzewo zachowań | Wykonuje kolejne ticki; wybiera retry, recovery albo przerwanie | Czy po naprawie udało się wykonać pierwotne zadanie |
| Obserwacja | Aktualizuje stan, np. dostępność kuchenki | Jakie skutki rzeczywiście miała nieudana akcja |

Tutaj `ActionExecutor` jest wykonawcą działania. Executor ROS z poprzednich ćwiczeń obsługuje callbacki; to inna rola.

W ćwiczeniu 12 `execute(state, action)` od razu wylicza **przewidywany kolejny stan**.
Tutaj używamy go przy starcie tylko do sprawdzenia legalności, a zapisujemy kolejny stan dopiero po sukcesie.
Wysyłamy raz, czekamy przez kilka ticków i zapisujemy efekt raz.

**Przewidź:** czy po samym `start(move)` położenie powinno już zmienić się na `kitchen`?

In [ ]:
initial = State()
executor = ActionExecutor(initial)
move = Action('move', 'kitchen')
executor.start(move, Outcome(success=True, running_ticks=2))
print('Po wysłaniu:', executor.status.value, executor.state.location, executor.state.battery)
assert executor.state == initial

for tick in range(1, 4):
    status = executor.tick()
    print(f'Tick {tick}: {status.value}, miejsce={executor.state.location}, bateria={executor.state.battery}')
    if tick <= 2:
        assert status is Status.RUNNING and executor.state == initial
assert status is Status.SUCCESS and executor.state == execute(initial, move)
committed = executor.state
assert executor.tick() is Status.SUCCESS and executor.state == committed
print('Kolejny tick nie naliczył ruchu ani zużycia baterii po raz drugi.')

**Oczekiwany ślad:** dwa razy `RUNNING` ze stanem `dock, 8`, potem `SUCCESS` ze stanem `kitchen, 6`.
`running_ticks=2` oznacza dwa ticki oczekiwania; odpowiedź pojawia się w trzecim.

**Eksperyment:** zmień tę liczbę na 4, wydłuż pętlę do pięciu ticków i zmień warunek `tick <= 2` na `tick <= 4`.
Czy czas oczekiwania powinien zwiększyć modelowy koszt przejazdu? W tym modelu koszt nie jest czasem.

## 2. Lokalna ponowna próba: to samo działanie

Zasymulujemy przejściowy błąd wykonawcy. Stan świata pozostaje taki sam, działanie jest nadal legalne.
Executor dopuszcza **jedną** lokalną ponowną próbę. Nie generuje nowej listy działań.
Drugi błąd wyczerpuje limit; nieograniczone ponawianie ukrywałoby problem.

**Założenie modelu:** nieudane działanie nie ma częściowych efektów fizycznych.
Po błędzie prawdziwej nawigacji robot może stać gdzie indziej — wtedy najpierw potrzebna jest obserwacja.
Nie zakładaj, że ponowienie operacji z efektem ubocznym jest zawsze bezpieczne.

In [ ]:
def finish_action(executor, max_ticks=20):
    for _ in range(max_ticks):
        status = executor.tick()
        if status is not Status.RUNNING:
            return status
    executor.cancel()
    raise TimeoutError('Działanie nie zakończyło się w limicie ticków')

retry_demo = ActionExecutor(State())
retry_demo.start(move, Outcome(success=False, running_ticks=1, message='Przejściowy błąd'))
assert finish_action(retry_demo) is Status.FAILURE
assert retry_demo.state == State() and retry_demo.can_retry
print('Po błędzie:', retry_demo.state, 'ponowna próba dostępna:', retry_demo.can_retry)
retry_demo.retry(Outcome(success=True, running_ticks=2))
assert finish_action(retry_demo) is Status.SUCCESS
assert retry_demo.dispatches == [move, move]
assert retry_demo.state.battery == 6
print('Wysłane działania:', retry_demo.dispatches)
print('Bateria zmalała raz:', retry_demo.state.battery)

## 3. Nowa obserwacja: stary plan przestaje obowiązywać

Po dojechaniu do kuchni próbujemy przygotować pierwszą porcję. Wykonawca zgłasza awarię kuchenki.
Przygotowanie nie zostało potwierdzone, więc posiłek nadal jest `ordered`.
Obserwacja zmienia `cooker_ok`, zachowując miejsce i baterię. Ponawianie `prepare` nie przywróci sprawności kuchenki wymaganej do rozpoczęcia tej akcji.

Odrzucamy pozostały stary plan i liczymy nowy **z obserwowanego stanu**.
Powrót do `State()` zgubiłby zużytą baterię i miejsce robota. Plan naprawczy zaczyna się od `repair`.

In [ ]:
at_kitchen = execute(State(), move)
fault_observation = replace(at_kitchen, cooker_ok=False)
fault_demo = ActionExecutor(at_kitchen)
fault_demo.start(Action('prepare', '1'), Outcome(
    success=False, running_ticks=2, observed=fault_observation, message='Awaria kuchenki'))
assert finish_action(fault_demo) is Status.FAILURE
assert fault_demo.state == fault_observation
assert fault_demo.state.meals == ('ordered', 'ordered') and not fault_demo.can_retry
recovery_plan, recovery_cost = solve(fault_demo.state)
assert recovery_plan[0] == Action('repair') and recovery_cost == 28
print('Obserwowany stan:', fault_demo.state)
print('Pierwszy krok:', recovery_plan[0], 'koszt pozostałej pracy:', recovery_cost)
for action in recovery_plan:
    fault_demo.start(action, Outcome(running_ticks=1))
    assert finish_action(fault_demo) is Status.SUCCESS
    print(action.name, action.target, '->', fault_demo.state.location, fault_demo.state.meals)
assert goal(fault_demo.state)
print('SUCCESS: obie porcje podane, kelner wrócił do bazy (dock).')

Awaria została zapisana w obserwowanym stanie; efekt `prepare` nie został zastosowany.
Obserwacja innej baterii też unieważnia założenia starego planu, nawet jeśli pierwszy ruch nadal jest legalny.

Jeśli obserwowany stan zmieni się **podczas** oczekiwania, `observe(...)` unieważnia oczekujący wynik.
Tutaj nie istnieje prawdziwy serwer do anulowania. Adapter ROS musi dodatkowo uzyskać potwierdzenie anulowania
i odrzucać spóźnione wyniki poprzedniego celu.

In [ ]:
pending = ActionExecutor(at_kitchen)
pending.start(Action('prepare', '1'), Outcome(success=True, running_ticks=3))
assert pending.tick() is Status.RUNNING
pending.observe(fault_observation)
assert pending.tick() is Status.FAILURE and pending.state == fault_observation
print('Spóźniony wynik sukcesu nie zmienił stanu posiłków na ready:', pending.state.meals)

**Brak planu też jest wynikiem.** Przy zerowej baterii z dala od ładowarki żadne legalne działanie nie doprowadzi do celu. Nie zgłaszamy sukcesu i nie uruchamiamy bez końca tego samego planera. Potrzebna jest interwencja lub nowe fakty.

In [ ]:
stranded_state = State(location='table_1', battery=0)
no_plan, no_cost = solve(stranded_state)
assert no_plan is None and no_cost is None and not goal(stranded_state)
print('BLOCKED: brak wykonalnego planu z obserwowanego stanu', stranded_state)

## 4. Twoje zadanie: wybór recovery

Napisz `recovery_policy(executor)`. Funkcja tylko podejmuje decyzję: **nie** zmienia executora ani nie wywołuje jego kolejnych ticków.
Przyjmujemy, że executor ma status `FAILURE`; dla innego statusu zgłoś `ValueError`.
Zwróć słownik z polami `mode`, `plan`, `cost`:

| Sytuacja | `mode` | `plan` | `cost` |
| --- | --- | --- | --- |
| Celowo anulowano (`failure_reason == 'cancelled'`) | `stopped` | `None` | `None` |
| `can_retry` jest prawdziwe | `retry` | Lista z tym samym `executor.action` | `None` |
| Lokalna próba niedostępna; `solve(executor.state)` znajduje plan | `replan` | Nowy plan | Koszt zwrócony przez planer |
| Brak wykonalnego planu | `blocked` | `None` | `None` |

Lokalny retry wykonujemy przez `executor.retry(...)`. Przeplanowanie zaczynamy przez `start(...)`
dla działań z nowego planu. Pusta lista jest poprawnym planem, gdy obserwowany stan już spełnia cel;
**brak planu to `None`**, nie `[]`.

<details><summary>Wskazówka</summary>
Najpierw sprawdź status i celowe anulowanie, potem can_retry. Dopiero potem wywołaj solve na executor.state.
Nie używaj nowego State() zamiast obserwowanego stanu. Nie uruchamiaj retry w funkcji podejmującej decyzję.
</details>

In [ ]:
def recovery_policy(executor):
    # TODO: wybierz ograniczoną lokalną próbę, przeplanowanie albo zatrzymanie.
    raise NotImplementedError

In [ ]:
def failed_action(initial, action, observed=None):
    instance = ActionExecutor(initial)
    instance.start(action, Outcome(success=False, running_ticks=0, observed=observed))
    assert instance.tick() is Status.FAILURE
    return instance

try:
    transient = failed_action(State(), move)
    assert recovery_policy(transient) == {'mode': 'retry', 'plan': [move], 'cost': None}
    assert transient.status is Status.FAILURE and len(transient.dispatches) == 1

    broken = failed_action(at_kitchen, Action('prepare', '1'), fault_observation)
    decision = recovery_policy(broken)
    assert decision['mode'] == 'replan' and decision['cost'] == 28
    assert decision['plan'][0] == Action('repair')
    checked = broken.state
    for action in decision['plan']:
        checked = execute(checked, action)
    assert goal(checked) and broken.state == fault_observation

    stranded = failed_action(State(), move, State(location='table_1', battery=0))
    assert recovery_policy(stranded) == {'mode': 'blocked', 'plan': None, 'cost': None}

    already_done = failed_action(State(), move, State(meals=('served', 'served')))
    assert recovery_policy(already_done) == {'mode': 'replan', 'plan': [], 'cost': 0}

    changed_energy = failed_action(State(), move, State(battery=4))
    assert recovery_policy(changed_energy)['mode'] == 'replan'
    assert changed_energy.state.battery == 4

    exhausted = failed_action(State(), move)
    exhausted.retry(Outcome(success=False, running_ticks=0))
    assert exhausted.tick() is Status.FAILURE
    assert recovery_policy(exhausted)['mode'] == 'replan'
    assert len(exhausted.dispatches) == 2

    stopped = ActionExecutor(State())
    stopped.start(move)
    stopped.cancel()
    assert recovery_policy(stopped) == {'mode': 'stopped', 'plan': None, 'cost': None}

    for instance in (ActionExecutor(), retry_demo):
        try:
            recovery_policy(instance)
        except ValueError:
            pass
        else:
            raise AssertionError('Polityka wymaga statusu FAILURE')
except NotImplementedError:
    print('TODO: implement recovery_policy; zadanie nie jest jeszcze zaliczone')
else:
    print('PASS: retry, replan, blocked i świadome zatrzymanie rozróżnione')

## 5. Jak podłączyć to do drzewa z ćwiczeń 9–11?

| Kontrakt liścia BT | Odpowiednik w modelu |
| --- | --- |
| Wysłanie celu przy wejściu (`initialise`) | `executor.start(action, outcome)` raz |
| `update()` podczas oczekiwania | `executor.tick()` → `RUNNING` |
| Potwierdzony wynik | `SUCCESS` albo `FAILURE`; efekty tylko po `SUCCESS` |
| `terminate(INVALID)` po przerwaniu | `executor.cancel()`; adapter ROS dodatkowo czeka na końcowy wynik anulowania |
| Lokalna gałąź recovery z ćwiczenia 10 | Jedna ponowna próba tego samego legalnego działania |
| Zmiana warunków świata | Odrzucenie starego planu i `solve(observed_state)` |

W [ćwiczeniu 9](9.%20Behavior%20Trees.ipynb) pamięć sekwencji chroni postęp liścia `RUNNING`,
a reaktywne priorytety pozwalają go przerwać.
[Ćwiczenie 10](10.%20Behavior%20Trees%20-%20ROS%20Applications.ipynb) dodaje własność celu i zasobów
oraz ograniczone recovery. [Ćwiczenie 11](11.%20Behavior%20Trees%20with%20Nav2%20Helper%20Demo.ipynb)
pokazuje czekanie na wynik Nav2: **przyjęcie celu to jeszcze nie dojazd**.
W restauracji `move` jest symbolem. Przyszły adapter zastąpi `Outcome` wynikiem Nav2,
a `prepare` potwierdzeniem wykonawcy kuchni.

**Projekt dla chętnych:** napisz liść `py_trees.behaviour.Behaviour` opakowujący executor.
Wysyłaj raz przy wejściu, mapuj statusy w `update()`, sprzątaj przy `INVALID`.
Przetestuj opóźnienie, błąd, przerwanie, spóźniony sukces i ponowne wejście.
Dwa wykonania nie mogą jednocześnie posiadać tego samego celu nawigacyjnego.

**Sprawdź rozumienie:** dlaczego zaplanowane `serve` nie może od razu ustawiać porcji na `served`?
Kiedy retry jest bezpieczny, a kiedy trzeba ponownie zaobserwować świat?
Co zrobić, gdy naprawa się udała, lecz ponowiona akcja znowu zwróciła błąd?
Jak odróżnisz brak planu od ukończenia wszystkich celów?